언어의미와 정보 수업을 위한 코드들

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 필요한 라이브러리 불러오기
import csv
import nltk
import pandas as pd
import numpy as np
import spacy
#https://spacy.io/usage/  pip install --upgrade pip 후에 pip install -U spacy 그리고 영어 모델 다운로드 python -m spacy download en_core_web_sm
from nltk.tokenize import word_tokenize
import string


In [ ]:
#01 텍스트를 스트링 형태로 불러오기 위한 함수 정의 : 영어 또는 프랑스어
def load_book(book_path, lower=False):
    '''
    Reads in a novel from a .txt file, and returns it in (optionally
    lowercased) string form

    Parameters
    ----------
    book_path : string (required)
        path to txt file containing full text of book to be analysed
    lower : bool (optional)
        If True, the returned string will be lowercased;
        If False, the returned string will retain its original case formatting.

    Returns
    -------
    book : string
        book in string form
    '''
    with open(book_path, encoding='utf-8') as f:
        book = f.read()
    if lower:
        book = book.lower()
    return book

In [ ]:
book_path = "/content/drive/MyDrive/Classroom/Copy of Harry Potter and the Sorcerer.txt"

# UTF-8 디코딩 에러를 방지하기 위해 errors='ignore' 옵션을 주어 파일을 안전하게 읽어옵니다.
try:
    book = load_book(book_path, lower=False)
except UnicodeDecodeError:
    # load_book 함수를 우회하여 오류 방지 처리를 하고 직접 읽어옵니다.
    with open(book_path, encoding='utf-8', errors='ignore') as f:
        book = f.read()

In [ ]:
book

In [ ]:
#02 텍스트를 문장 단위로 자르기
def get_sentence_sequences(book):
    '''
    Splits a book into its constituent sentences

    Parameters
    ----------
    book : string (required)
        book in string form

    Returns
    -------
    sentences : list
        list of strings, where each string is a sentence in the novel as
        interpreted by NLTK's tokenize() function.
    '''
    detector = nltk.data.load('tokenizers/punkt/english.pickle')
    sentences = detector.tokenize(book)
    return sentences

In [ ]:
# nltk.download() 가 먼저 필요할 수 있음. - 모든 nltk 데이터 자료 다운받기
nltk.download("punkt_tab") # punkt_tab 리소스 명시적 다운로드
sequences = get_sentence_sequences(book)

In [ ]:
len(sequences)

In [ ]:
#03 캐릭터 추출하기
# 읽어 들인 텍스트의 단어 수 점검. 공란을 기준으로 자르기, 중복제거
print(len(book.split()))

remove_punctuation = lambda s: s.translate(str.maketrans('', '', string.punctuation+'\''))
words = [remove_punctuation(p) for p in book.split()]

unique_words = list(set(words))
print(len(unique_words))

In [ ]:
#04 텍스트 단어들 중에서 고유명사 가져오기
#spaCy이용
nlp = spacy.load('en_core_web_sm')

candidates = [word.text for word in nlp(' '.join(unique_words)) if word.pos_ == 'PROPN']
print(len(candidates))

#2글자 미만인 것들은 제거
candidates = [c for c in candidates if len(c) > 2]
print(len(candidates))

In [ ]:
# 05 인명 후보자들 가져오기
candidates = [c for c in candidates if c.istitle()]
print(len(candidates))

# 06 소유격이 붙은 것 제거
candidates = [c for c in candidates if not (c[-1] == 's' and c[:-1] in candidates)]
print(len(candidates))

# 07 영어 불용어 제거
import nltk
nltk.download('stopwords')
stopwords = nltk.corpus.stopwords.words('english')
candidates = list(set([c.title() for c in [c.lower() for c in candidates]]) - set(stopwords))
print(len(candidates))
candidates[:10]

In [ ]:
# 08 캐릭터 이름을 튜플 형태로 만들기 / csv 파일로 저장하기
import os

characters = [tuple([character + ' ']) for character in set(candidates)]
output_dir = '/content/drive/MyDrive/data'

# 폴더가 존재하지 않으면 자동으로 생성합니다.
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

with open(os.path.join(output_dir, 'characters_hpss.csv'), 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerows(characters)

In [ ]:
characters[0]

In [ ]:
# 09 문장들속에 나타나는 캐릭터 연결 추출하기
def find_connections(sequences, characters):
    '''
    Takes a novel and its character list and counts instances of each character
    in each sequence.

    Parameters
    ----------
    sequences : list (required)
        list of substrings representing the novel to be analysed
    characters : list (required)
        list of charater names (as tuples)

    Returns
    -------
    df : pandas.DataFrame
        columns = character names
        indexes = sequences
        values  = counts of instances of character name in sequence
    '''
    if any(len(names) > 1 for names in characters):
        df = pd.DataFrame({str(character):
                           {sequence: sum([sequence.count(name) for name in character])
                            for sequence in sequences}
                           for character in characters})
    else:
        characters = [c[0] for c in characters]
        df = pd.DataFrame([[sequence.count(character)
                            for character in characters]
                           for sequence in sequences],
                          index=sequences,
                          columns=characters)
    return df



In [ ]:
df = find_connections(sequences, characters)

In [ ]:
df.head()

In [ ]:
# 10 캐릭터들의 동시발생 계산하기
def calculate_cooccurence(df):
    '''
    Uses the dot product to calculate the number of times two characters appear
    in the same sequences. This is the core of the bookworm graph.

    Parameters
    ----------
    df : pandas.DataFrame (required)
        columns = character names
        indexes = sequences
        values  = counts of instances of character name in sequence

    Returns
    -------
    cooccurence : pandas.DataFrame
        columns = character names
        indexes = character names
        values  = counts of character name cooccurences in all sequences
    '''
    characters = df.columns.values
    cooccurence = df.values.T.dot(df.values)
    np.fill_diagonal(cooccurence, 0)
    cooccurence = pd.DataFrame(cooccurence, columns=characters, index=characters)
    return cooccurence

cooccurence = calculate_cooccurence(df)

In [ ]:
cooccurence

In [ ]:
# 11 상호관계 추출 - NetworkX에 필요
def get_interaction_df(cooccurence, threshold=0):
    '''
    Produces an dataframe of interactions between characters using the
    cooccurence matrix of those characters. The return format is directly
    analysable by networkx in the construction of a graph of characters.

    Parameters
    ----------
    cooccurence : pandas.DataFrame (required)
        columns = character names
        indexes = character names
        values  = counts of character name cooccurences in all sequences
    threshold : int (optional)
        The minimum character interaction strength needed to be included in the
        returned interaction_df

    Returns
    -------
    interaction_df : pandas.DataFrame
        DataFrame enumerating the strength of interactions between charcters.
        source = character one
        target = character two
        value = strength of interaction between character one and character two
    '''
    rows, columns = np.where(np.triu(cooccurence.values, 1) > threshold)

    return pd.DataFrame(np.column_stack([cooccurence.index[rows],
                                         cooccurence.columns[columns],
                                         cooccurence.values[rows, columns]]),
                        columns=['source', 'target', 'value'])

import networkx as nx
interaction_df = get_interaction_df(cooccurence, threshold=2)
interaction_df.sample(5)

In [ ]:
interaction_df[:50]

In [ ]:
# 12 캐릭터 상호관계 시각화
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (24,18)

G = nx.from_pandas_edgelist(interaction_df,
                             source='source',
                             target='target')
pos = nx.spring_layout(G, k=10.0, scale=10.0)
nx.draw(G, with_labels=True)
plt.show()

In [ ]:
#PyVis 이용
!pip install pyvis

In [ ]:
from pyvis.network import Network
net = Network(notebook=True, cdn_resources='in_line')
net.from_nx(G)
net.show('example.html')

from IPython.display import HTML

with open('example.html', 'r') as f:
    html_content = f.read()

HTML(html_content)

In [ ]:
# 노드별 연결 정도 (Degree Centrality) 확인
degree_centrality = nx.degree_centrality(G)

# 연결 정도가 높은 상위 10개 노드 출력
print("상위 10개 노드의 연결 정도:")
for node, centrality in sorted(degree_centrality.items(), key=lambda item: item[1], reverse=True)[:10]:
    print(f"Node: {node}, Centrality: {centrality:.4f}")

* 셀 [26](실제 노트북의 b732da26 및 e42eee09 셀)에서 수행한 연결 중심성(Degree Centrality) 분석 결과를 바탕으로 이 그래프에서 알 수 있는 핵심적인 사실들은 다음과 같습니다.

1. 연결 중심성(Degree Centrality)의 의미
연결 중심성은 한 노드(인물)가 그래프 내에서 얼마나 많은 다른 노드들과 직접적인 연결(관계)을 맺고 있는지를 나타내는 지표입니다.
즉, 이 소설(해리포터와 마법사의 돌) 내에서 가장 마당발이거나 다양한 인물군과 물리적으로 자주 얽히는 주도적인 인물이 누구인지 직관적으로 보여줍니다.
2. 상위 노드 분석을 통해 알 수 있는 특징
출력된 상위 10개 노드의 연결 정도(Centrality) 수치는 다음과 같습니다.

Harry (0.3258): 주인공인 해리가 가장 압도적인 연결 중심성 값을 가집니다. 전체 등장인물 중 약 32.6%의 인물들과 직접적으로 같은 문장 안에서 언급되었음을 뜻하며, 이야기의 절대적인 중심축임을 증명합니다.
Magical (0.2348), Gryffindor (0.1742): 인물이 아닌 고유 명사들도 높은 중심성을 보입니다. 이는 작중에서 다양한 인물들이 '마법(Magical)'적인 상황에 놓이거나 '그리핀도르(Gryffindor)' 기숙사 소속원으로서 빈번하게 엮이기 때문입니다.
Quaffle (0.1439), Bludger (0.1439): 퀴디치 경기 용어들인 콰플과 블러저가 상위에 랭크되어 있습니다. 이는 퀴디치 경기 장면이나 대화에서 해리를 비롯해 수많은 등장인물(우드, 플린트, 안젤리나 등)이 한꺼번에 동시에 언급되었기 때문에 그래프 상에서 다수의 인물과 직접 연결되는 강력한 허브 역할을 하고 있음을 알 수 있습니다.
Professor (0.1061), Snape (0.0833), Hermione (0.0833), Ron (0.0606): 해리의 주변 핵심 인물(론, 헤르미온느, 스네이프 교수 등)이 그 뒤를 이어 높은 중심성을 차지합니다. 이들이 해리뿐만 아니라 학교 내의 다른 여러 인물과도 넓은 교우 관계 및 대립 관계를 형성하고 있음을 보여줍니다.
3. 시각적 하이라이트 효과 (셀 e42eee09 관련)
분석 결과를 바탕으로 연결 중심성이 높은 상위 5개 핵심 노드(Harry, Magical, Gryffindor, Quaffle, Bludger)를 빨간색 대형 노드로 강조하고, 나머지 일반 노드들을 청록색 소형 노드로 시각화함으로써 전체 인물 관계망에서 스토리의 전개를 주도하는 핵심 키워드와 인물들이 무엇인지 한눈에 파악할 수 있게 도와줍니다.

In [ ]:
# 상위 5개 노드 추출
top_5_nodes = sorted(degree_centrality.items(), key=lambda item: item[1], reverse=True)[:5]
top_5_node_names = [node for node, centrality in top_5_nodes]

# PyVis 네트워크의 노드 스타일 변경
for node in net.nodes:
    if node['id'] in top_5_node_names:
        node['color'] = '#FF0000' # 빨간색으로 변경
        node['size'] = 30         # 크기를 더 크게 변경
    else:
        node['color'] = '#008080' # 나머지 노드는 다른 색상으로 유지
        node['size'] = 15

# 변경된 네트워크 시각화 저장 및 표시
net.show('highlighted_example.html')

from IPython.display import HTML

with open('highlighted_example.html', 'r') as f:
    html_content_highlighted = f.read()

HTML(html_content_highlighted)

### 가중치 기반 네트워크 분석

In [ ]:
# interaction_df에서 'value' 열을 가중치로 사용하여 가중치 그래프 생성
G_weighted = nx.from_pandas_edgelist(interaction_df,
                                     source='source',
                                     target='target',
                                     edge_attr='value',
                                     create_using=nx.Graph)

# 가중치 기반 연결 중심성 (Weighted Degree Centrality) 계산
# `degree()` 메서드는 weight='value' 인자를 사용하여 가중치 합계를 반환할 수 있습니다.
weighted_degree_centrality = dict(G_weighted.degree(weight='value'))

# 가중치 기반 연결 중심성이 높은 상위 10개 노드 출력
print("상위 10개 노드의 가중치 기반 연결 정도:")
for node, centrality in sorted(weighted_degree_centrality.items(), key=lambda item: item[1], reverse=True)[:10]:
    print(f"Node: {node}, Weighted Degree Centrality: {centrality:.2f}")

* 셀 [27](실제 노트북의 844c5192 셀)에서 수행한 가중치 기반 연결 중심성(Weighted Degree Centrality) 분석 결과를 통해 알 수 있는 핵심적인 사실들은 다음과 같습니다.

1. 가중치 기반 연결 중심성(Weighted Degree Centrality)의 의미
이전의 일반 연결 중심성(Degree Centrality)은 인물들이 '단순히 얼마나 다양한 사람들과 아는가(연결된 선의 수)'를 측정했습니다.
반면, 가중치 기반 연결 중심성은 단순한 연결 관계의 수를 넘어 **'그 연결이 얼마나 자주, 긴밀하게 일어났는가(선들의 가중치/동시 언급 빈도의 합)'**를 반영합니다.
따라서 이를 통해 작중에서 진짜로 대화가 많고 실질적인 인터랙션(상호작용) 강도가 높은 핵심 주역들이 누구인지 훨씬 더 정확하게 가려낼 수 있습니다.
2. 가중치 분석 결과 해석 및 단순 중심성과의 비교
분석 결과로 출력된 상위 10개 노드의 가중치 수치는 다음과 같습니다.

Harry (494.00): 가중치 기준에서도 압도적인 1위입니다. 다른 캐릭터들과 단순히 넓게 알 뿐만 아니라, 깊고 조밀하게 끊임없이 소통하고 있음을 극명하게 보여줍니다.
Ron (156.00) & Hermione (127.00): 단순 연결 중심성 순위에서는 론(10위, 0.0606)과 헤르미온느(8위, 0.0833)가 상대적으로 낮게 나타났었습니다. 하지만 가중치 분석에서는 각각 2위와 4위로 급상승했습니다. 이는 이 둘이 맺고 있는 관계의 다양성은 해리보다 좁을지 몰라도, 해리를 비롯한 주역들과의 실제 대화 및 함께 등장하는 빈도(상호작용 깊이)가 엄청나게 높기 때문입니다. 진짜 핵심 주역의 면모를 여기서 확인할 수 있습니다.
Uncle (141.00) & Vernon (119.00) & Dudley (90.00) & Aunt (87.00): 해리의 이모부(버논), 이모(페투니아), 사촌(두들리) 등 더즐리 가족들이 상위권에 대거 등장합니다. 소설 초반부 더즐리 가족과의 긴밀하고 반복적인 갈등 구조가 텍스트에 깊이 각인되어 있음을 가중치를 통해 알 수 있습니다.
단어 노드들의 하락 (Magical, Gryffindor 등): 단순 중심성에서 2, 3위를 기록했던 Magical과 Gryffindor는 가중치 순위에서 각각 8위와 7위로 내려앉았습니다. 이 단어들은 다양한 인물과 산발적으로 넓게 엮이지만, 특정 인물들 간의 밀도 높은 서사나 깊은 상호작용처럼 집중적인 고빈도 연결은 아니었다는 것을 의미합니다.
요약하자면
가중치 기반 분석을 통해 **단순히 넓은 마당발인 캐릭터(예: 스포츠 용어 등)**와 **진짜로 이야기 속에서 긴밀한 상호작용을 하며 서사를 이끌어가는 실제 주인공 집단(Harry, Ron, Hermione, 그리고 더즐리 가문)**을 뚜렷하게 구별하여 파악할 수 있습니다.

### 가중치 그래프 시각화 (노드 크기를 가중치 기반 연결 정도로 반영)

In [ ]:
plt.figure(figsize=(24, 18))

# 노드 크기를 가중치 기반 연결 정도에 비례하도록 설정
node_sizes = [v * 100 for v in weighted_degree_centrality.values()]

# 노드 색상 지정 (선택적으로)
node_colors = ['skyblue' for _ in G_weighted.nodes()]

# 레이아웃 설정 (더 보기 좋게)
pos = nx.spring_layout(G_weighted, k=0.15, iterations=50) # k 값과 iterations 조정

nx.draw_networkx_nodes(G_weighted, pos, node_size=node_sizes, node_color=node_colors, alpha=0.8)
nx.draw_networkx_edges(G_weighted, pos, width=[(d['value'] * 0.5) for (u, v, d) in G_weighted.edges(data=True)], alpha=0.5, edge_color='gray')
nx.draw_networkx_labels(G_weighted, pos, font_size=10, font_color='black')

plt.title('가중치 기반 네트워크 시각화 (노드 크기 = 가중치 연결 정도)', size=20)
plt.axis('off')
plt.show()

In [ ]:
# 12-1 draw spring
nx.draw_spring(G, with_labels=True)

In [ ]:
# 0___ 추출된 캐릭터 파일 읽어오기
def load_characters(charaters_path):
    '''
    Reads in a .csv file of character names

    Parameters
    ----------
    charaters_path : string (required)
        path to csv file containing full list of characters to be examined.
        Each character should take up one line of the file. If the character is
        referred to by multiple names, nicknames or sub-names within their
        full name, these should be split by commas, eg:
        Harry, Potter
        Lord, Voldemort, You-Know-Who
        Giant Squid

    Returns
    -------
    characters : list
        list of tuples naming characters in text
    '''
    with open(charaters_path) as f:
        reader = csv.reader(f)
        characters = [tuple(name.lower()+' ' for name in row) for row in reader]
    return characters

In [ ]:
characters_path='/content/drive/MyDrive/data/characters_hpss.csv'
characters_list = load_characters(characters_path)
characters_list

# 캐릭터 사이의 관계에 대한 통계 수치들 뽑아보기

In [ ]:
# 인물별 중요도, 즉 핵심 노드 정도
pd.Series(nx.pagerank(G)).sort_values(ascending=False)[:5]

In [ ]:
a, b = nx.hits(G)
pd.Series(a).sort_values(ascending=False)[:5]

* 수업 코드 내 [33]번 셀(mp0OS5lsx-mb)과 [34]번 셀(LHF-m7Qtywdx)은 네트워크 분석에서 노드의 중요도(Centrality)를 측정하는 서로 다른 두 가지 대표적인 알고리즘을 사용하고 있습니다. 그 차이점을 알기 쉽게 설명해 드리겠습니다.

1. [33]번 셀: PageRank (페이지랭크)
코드: pd.Series(nx.pagerank(G)).sort_values(ascending=False)[:5]
개념: 구글 검색 엔진의 기반이 된 알고리즘으로, **"중요한 노드로부터 링크를 많이 받는 노드가 진짜 중요한 노드다"**라는 무작위 유저의 이동(Random Walk) 모델을 기반으로 합니다.
특징: 단순히 연결선이 많다고 해서 점수가 높아지는 것이 아닙니다. 예를 들어 중요도가 매우 높은 Harry나 Dumbledore 같은 거물급 노드와 단 한 번이라도 직접 연결되어 있다면, 그 영향력(점수)이 전파되어 자신도 높은 점수를 얻게 됩니다.
상위 결과: Magical (0.102), Harry (0.074), Gryffindor (0.028) 순
2. [34]번 셀: HITS (Hits 알고리즘 - Hubs and Authorities)
코드: a, b = nx.hits(G); pd.Series(a).sort_values(ascending=False)[:5]
개념: 웹페이지를 두 가지 역할, 즉 정보를 많이 담고 있는 **권위자(Authority)**와 좋은 정보들을 가리키는 **허브(Hub)**로 나누어 상호 전파 방식으로 점수를 매기는 알고리즘입니다.
특징:
Authority(권위자, 출력된 a): 유용한 정보를 많이 담은 페이지로, 좋은 Hub들로부터 링크를 많이 받는 노드입니다.
Hub(허브, 출력된 b): 좋은 Authority들을 많이 가리키고 연결해주는 가이드 역할을 하는 노드입니다.
이 코드는 인물 관계망에서 **'누가 가장 대표적인 권위 있는 중심 허브(Authority) 역할을 하고 있는가'**를 보여줍니다.
상위 결과: Harry (0.059), Gryffindor (0.058), Bludger (0.044) 순
3. 핵심 차이점 요약
알고리즘의 메커니즘 차이:

PageRank는 네트워크 전체를 흘러 다니는 수치(유동성/확률)가 특정 노드에 머무를 확률을 계산합니다. 한 방향의 일방적인 전파 흐름에 강합니다.
HITS는 'Hub'와 'Authority'가 서로의 점수를 높여주는 상호 보완적 피드백(상호 강화) 구조입니다. 연결성이 조밀한 상호 연결 그룹(예: 그리핀도르 소속원, 퀴디치 팀원들)의 끈끈한 네트워크를 잡아내는 데 특화되어 있습니다.
분석 결과의 차이:

PageRank 결과에서는 Magical이 1위, Harry가 2위를 차지했습니다. Magical 단어는 책 전체에 걸쳐 넓은 맥락에서 고르게 영향력을 전파하고 있기 때문입니다.
HITS 결과에서는 실제 이야기의 실질적 구심점인 Harry가 1위, 소속 기숙사인 Gryffindor가 2위로 올라옵니다. 인물과 커뮤니티가 촘촘하게 엮여 있는 서사적 관계 집단(Clique)을 더 민감하게 반영하기 때문입니다.

In [ ]:
list(nx.enumerate_all_cliques(G))[-1]

In [ ]:
comms = nx.communicability(G)

#comms
print(comms["Vernon "]["Dumbledore "])
print(comms["Harry "]["Hermione "])

In [ ]:
nx.dijkstra_path(G,
                 source="Hedwig ",
                 target="Flamel ")

* 노트북의 [35], [36], [37]번 셀에서 수행하고 있는 분석 내용과 이를 통해 파악할 수 있는 유의미한 정보들을 요약하여 설명해 드리겠습니다.

1. [35]번 셀: 클리크(Clique) 분석
코드: list(nx.enumerate_all_cliques(G))[-1]
결과: ['Harry ', 'Aunt ', 'Vernon ', 'Uncle ', 'Petunia ', 'Dudley ']
개념 설명:
**클리크(Clique, 완벽한 부분그래프)**란 소속된 모든 노드(인물)들이 서로 빠짐없이 전부 1:1로 직접 연결되어 있는 완전히 닫힌 상호 관계 그룹을 의미합니다.
코드의 [-1]은 전체 클리크 집합 중 **가장 크기가 큰 최대 클리크(Maximal Clique)**를 추출한 것입니다.
알 수 있는 정보:
이 그래프에서 가장 강력하고 촘촘한 결속력을 가진 6인 그룹은 **해리(Harry)와 그의 이모네 가족(버논 이모부, 페투니아 이모, 사촌 두들리 등 더즐리 가문)**입니다.
이 집단은 단 한 명도 빠짐없이 서로 긴밀하게 상호작용하고 얽혀 있음을 입증하며, 소설의 극 초반부 및 해리의 가정 환경적 배경이 얼마나 밀도 높은 집단적 관계를 형성하고 있는지 알 수 있습니다.
2. [36]번 셀: 통신성(Communicability) 측정
코드: comms = nx.communicability(G)
개념 설명:
**통신성(Communicability)**은 네트워크에서 두 노드 사이에 정보나 영향력이 얼마나 쉽고 효율적으로 잘 흘러갈 수 있는지를 측정하는 지표입니다.
단순히 두 사람 사이의 직통 최단 거리뿐만 아니라, 우회할 수 있는 모든 간접적인 경로(모든 길이의 경로)의 가중치 합을 기반으로 계산됩니다.
알 수 있는 정보:
특정 캐릭터(예: 해리)로부터 다른 캐릭터(예: 헤르미온느, 스네이프 등)까지 영향력이 얼마나 빠르게 전달될 수 있는지를 상대적 점수로 파악할 수 있습니다.
예를 들어 직접 대화를 나누지 않는 관계라도 공통 지인(Hub)이 많다면 통신성 점수가 높게 계산되어, 이들이 이야기 흐름 상 직간접적으로 긴밀하게 묶여 있는 인물 구도라는 것을 알려줍니다.
3. [37]번 셀: 다익스트라(Dijkstra) 최단 경로 탐색
코드: nx.dijkstra_path(G, source="Hedwig ", target="Flamel ")
결과: ['Hedwig ', 'Harry ', 'Dumbledore ', 'Nicolas ', 'Flamel ']
개념 설명:
다익스트라 알고리즘은 그래프 상의 특정 출발 노드에서 도착 노드까지 도달하는 **가장 빠르고 효율적인 최단 경로(최단 인물 사슬)**를 추적하는 알고리즘입니다.
알 수 있는 정보:
접점이 거의 없어 보이는 **헤드위그(Hedwig, 올빼미)**에서 소설 속 전설적인 연금술사인 **니콜라스 플라멜(Nicolas Flamel)**까지 어떻게 인간관계망으로 연결되는지 계보를 보여줍니다.
헤드위그 → (주인) 해리 → (교장) 덤블도어 → (동업자) 니콜라스 → 플라멜로 이뤄지는 다리를 가교 삼아 소설 속 사건의 단서들이 어떻게 인물을 거쳐 소문이나 정보로 전파될 수 있는지를 지리학적 경로처럼 가시적으로 보여줍니다.
💡 총 요약
[35]: 소설 내에서 **가장 촘촘하고 끈끈하게 모두 엮여 있는 핵심 집단(더즐리 가문)**을 찾아냈습니다.
[36]: 정보와 소문이 인물 관계 네트워크 상에서 어떤 연결 경로들을 통해 효과적으로 전파되는지의 전파 용이성을 계산했습니다.
[37]: 전혀 무관해 보이는 극단에 있는 두 대상(헤드위그와 플라멜)이 몇 단계의 '아는 사람' 다리를 건너면 연결되는지 구체적인 인물 사슬 경로를 밝혀냈습니다.